# DuckPD Vector Search & Streaming Text Embeddings Walkthrough

This interactive notebook demonstrates how to use **DuckPD** for text embeddings and exact vector similarity search over remote and local Parquet datasets.

### Highlights
- **FastEmbed / ONNX runtime integration**: Seamlessly prepare and use quantized text embedding models locally on CPU.
- **Lazy remote streaming**: Scan Parquet datasets directly over HTTPS without downloading everything upfront.
- **In-engine text embedding**: Embed text columns lazily in bounded Arrow batches via `.embed_text()`.
- **Vector search API**: Query embedded datasets using `.vector.search_text()` with cosine distance, top-$k$ retrieval, and deterministic tie-breaking.

## 1. Imports and configuration

Define the remote source, a cache beside the notebook, the search query, and a revision-pinned embedding model. The cache path works whether the kernel starts in the repository root or in `demo/`.

In [ ]:
from pathlib import Path
from time import perf_counter

import duckpd as pd

DATA_URL = "https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"
DEMO_DIR = Path("demo") if Path("demo").is_dir() else Path(".")
EMBEDDED_DATA = DEMO_DIR / "nvidia-news-embedded.parquet"
QUERY = "AI chip demand and revenue growth"

MODEL = pd.embedding_model(
    "BAAI/bge-small-en-v1.5",
    revision="5c38ec7c405ec4b44b94cc5a9bb96e735b38267a",
    dimension=384,
)

print(f"DuckPD version: {pd.__version__}")
print(
    f"Embedding model: {MODEL.model} "
    f"(revision: {MODEL.revision[:12]}..., dimension: {MODEL.dimension})"
)
print(f"Embedded dataset: {EMBEDDED_DATA}")

## 2. Prepare the embedding model

Open a DuckPD session and prepare the text embedding model. Preparation initializes the local FastEmbed/ONNX backend and reports its available execution providers.

In [ ]:
session = pd.connect()

preparation_started = perf_counter()
prepared = session.prepare_embedding_model(MODEL)
preparation_seconds = perf_counter() - preparation_started

print(f"Backend: {prepared.backend} via {prepared.execution_providers}")
print(f"Model preparation time: {preparation_seconds:.3f}s")
print(f"Persisted model fingerprint: {MODEL.fingerprint}")

## 3. Load or Build Embedded Dataset

If a pre-embedded Parquet file exists locally, we load it directly. Otherwise, DuckPD lazily scans the remote dataset from Hugging Face, filters for NVIDIA (`NVDA`) news articles, computes text embeddings across the `title` and `description` columns in batches, and persists the result to Parquet.

In [ ]:
if EMBEDDED_DATA.exists():
    embedded = session.read_parquet(EMBEDDED_DATA)
    dataset_status = f"Loaded {EMBEDDED_DATA}"
else:
    build_started = perf_counter()
    news = session.read_parquet(DATA_URL)
    nvidia = news[news["symbol"] == "NVDA"]
    embedded = nvidia.embed_text(
        columns=["title", "description"],
        into="embedding",
        model=MODEL,
        batch_size=64,
        null_policy="empty",
    )
    embedded.write_parquet(EMBEDDED_DATA)
    build_seconds = perf_counter() - build_started
    dataset_status = (
        f"Created {EMBEDDED_DATA} from the remote archive "
        f"in {build_seconds:.3f} seconds"
    )
    embedded = session.read_parquet(EMBEDDED_DATA)

print(f"Embedding dataset: {dataset_status}")
print(f"Columns: {embedded.columns}")

## 4. Preview the source data

Inspect a few identifying columns with bounded materialization via `head()`. The large `embedding` vectors are intentionally omitted here; they remain available in the lazy frame for search.

In [ ]:
preview = embedded[["symbol", "title", "publisher", "publish_date"]].head(5)
preview

## 5. Run a vector similarity search

Use `.vector.search_text()` to embed the query and retrieve the five nearest rows from the `embedding` column by cosine distance. The `title` tie-breaker keeps equally scored matches deterministic.

In [ ]:
query_started = perf_counter()
matches = embedded.vector.search_text(
    QUERY,
    column="embedding",
    model=MODEL,
    metric="cosine",
    k=5,
    tie_breaker="title",
)[["symbol", "title", "publisher", "publish_date", "_distance"]]
result = matches.collect()
query_seconds = perf_counter() - query_started

print(f"Query: {QUERY!r}")
print(f"Query-to-response: {query_seconds:.3f} seconds")

## 6. Inspect the results

Lower cosine distance means greater semantic similarity. Display the ranked matches, then close the DuckPD session.

In [ ]:
display(result)
session.close()